# Couchbase: run, explain and profile

Start and initialize the Community Docker environment using the local database guide.
Install `uv sync --extra couchbase` and use this project's Python environment.
Set `COUCHBASE_USERNAME` / `COUCHBASE_PASSWORD` to the documented local credentials.
All examples are read-only and use the eight synthetic demo businesses.


In [ ]:
import os
from pprint import pprint

from couchbase.n1ql import QueryScanConsistency

from dbrtoolkit.couchbase import AsyncCouchbaseExecutor, CouchbaseQuery
from dbrtoolkit.workload import Workload, WorkloadEntry

# For the optional Enterprise environment, set COUCHBASE_URI to port 11211.
uri = os.environ.get("COUCHBASE_URI", "couchbase://127.0.0.1:11210?network=external")
options = {"read_only": True, "scan_consistency": QueryScanConsistency.REQUEST_PLUS}

city_query = CouchbaseQuery(
    "SELECT RAW b.name FROM business b WHERE city = $city ORDER BY b.name",
    named_parameters={"city": "Sydney"},
    query_context="demo._default",
)
count_query = CouchbaseQuery("SELECT RAW COUNT(*) FROM `demo`.`_default`.`business`")
workload = Workload(
    [WorkloadEntry(city_query, pattern_id=0), WorkloadEntry(count_query, pattern_id=1)]
)

## One query, three operations

`run` consumes rows and optionally retains them. `explain` returns a plan without
executing the original query. `profile` executes with diagnostic timing enabled;
it can retain rows too. Community returns basic metrics and a warning about missing
detailed profiling; Enterprise adds an operator tree. No extra EXPLAIN query is issued.


In [ ]:
async with await AsyncCouchbaseExecutor.from_uri(
    uri,
    username=os.environ["COUCHBASE_USERNAME"],
    password=os.environ["COUCHBASE_PASSWORD"],
    concurrency=2,
    continue_on_error=True,
    timeout_seconds=10,
) as executor:
    result = await executor.run(city_query, collect_results=True, query_kwargs=options)
    plan = await executor.explain(city_query, query_kwargs=options)
    measured = await executor.profile(city_query, collect_results=True, query_kwargs=options)
    runs = await executor.run_workload(workload, query_kwargs=options)
    plans = await executor.explain_workload(workload, query_kwargs=options)
    profiles = await executor.profile_workload(workload, query_kwargs=options)

print("Names:", result.documents)
print("Consumed rows:", result.row_count)
print("Same profile rows:", measured.documents == result.documents)
pprint(plan.raw["plan"], depth=3)
pprint(measured.raw["metrics"])
print("Warnings:", measured.warnings)

## Workload outcomes

All three workload methods return input-aligned lists. `None` means an entry was
never started. Expected database failures appear in `.error`; programming errors
still raise. With concurrent execution, already-running entries finish after an error.


In [ ]:
for name, results in (("run", runs), ("explain", plans), ("profile", profiles)):
    print(name, [(r.status, r.row_count) if r is not None else None for r in results])

# EXPLAIN rows describe plans, not query output, so they have no row_count.
assert plans[0].row_count is None
assert runs[0].row_count == profiles[0].row_count == 3
# SELECT RAW COUNT(*) returns one scalar row; it does not return eight rows.
assert runs[1].row_count == 1

## Inspect the native diagnostics

Metadata uses plain JSON values and milliseconds. The profile tree keeps Couchbase's
native operator names and counters. An emitted index item is not automatically a
MongoDB-style examined index entry. Common summary parsing is the next phase.

`REQUEST_PLUS` waits for fresh index contents; that wait affects measurements.
Try `NOT_BOUNDED` only when you accept potentially stale index contents.


In [ ]:
profile_tree = (measured.raw["profile"] or {}).get("executionTimings")
if profile_tree is None:
    print("No detailed operator profile; basic metrics are still available.")
else:
    pprint(profile_tree, depth=3)